# 03 · Evaluación y contraste

**Fase 4 — Semanas 17 a 21. Aquí se abre el conjunto de TEST por primera y única vez.**
Todo lo que sale de este cuaderno es el **Capítulo IV** de la tesis.

Pasos (PDF):
1. Evaluar el modelo profundo y la línea base sobre el **mismo** test.
2. Reportar exactitud, precisión, exhaustividad, **F1** y **AUC-ROC** + matriz de confusión.
3. Aplicar la **prueba de McNemar**: su valor *p* decide la hipótesis (no la diferencia de F1).
4. Criterio técnico: **F1 ≥ 0.90**.
5. Revisar a mano ~20 errores del modelo.

> Este cuaderno corre con un **DEMO** si aún no hay predicciones reales, para que veas
> cómo funciona toda la evaluación. Con datos reales, cargas los modelos y predices sobre test.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (classification_report, f1_score, accuracy_score,
                             confusion_matrix, roc_auc_score)
from statsmodels.stats.contingency_tables import mcnemar

SEED = 42
rng = np.random.default_rng(SEED)

RAIZ = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
PARTICIONES = RAIZ / 'data' / 'particiones'
MODELOS = RAIZ / 'modelos'
RESULTADOS = RAIZ / 'resultados'
RESULTADOS.mkdir(exist_ok=True)
CLASES = ['legitimo', 'fraudulento']
F1_MINIMO = 0.90   # criterio técnico de aceptación del PDF

## 1. Funciones de evaluación (se reutilizan para cada motor)

In [ ]:
def matriz(y, pred, titulo, archivo=None):
    cm = confusion_matrix(y, pred, labels=CLASES)
    fig, ax = plt.subplots(figsize=(4.2, 3.6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=CLASES, yticklabels=CLASES, cbar=False, ax=ax)
    ax.set_xlabel('Predicción'); ax.set_ylabel('Etiqueta real'); ax.set_title(titulo)
    plt.tight_layout()
    if archivo:
        plt.savefig(RESULTADOS / archivo, dpi=120)
    plt.show()

def prueba_mcnemar(y, pred_a, pred_b):
    """Compara dos modelos sobre las MISMAS observaciones."""
    y = np.asarray(y); pred_a = np.asarray(pred_a); pred_b = np.asarray(pred_b)
    a_ok = pred_a == y
    b_ok = pred_b == y
    tabla = [[int(np.sum(a_ok & b_ok)),  int(np.sum(a_ok & ~b_ok))],
             [int(np.sum(~a_ok & b_ok)), int(np.sum(~a_ok & ~b_ok))]]
    res = mcnemar(tabla, exact=False, correction=True)
    return tabla, res.pvalue

In [ ]:
def comparar(y, pred_prof, pred_base, score_prof=None, score_base=None, nombre='motor'):
    y = np.asarray(y)
    print(f'#################### {nombre.upper()} ####################\n')
    for etiqueta, pred in [('MODELO PROFUNDO', pred_prof), ('LÍNEA BASE', pred_base)]:
        print(f'----- {etiqueta} -----')
        print(classification_report(y, pred, digits=3, zero_division=0))

    f1_prof = f1_score(y, pred_prof, pos_label='fraudulento', zero_division=0)
    f1_base = f1_score(y, pred_base, pos_label='fraudulento', zero_division=0)
    print(f'F1 (fraudulento)  profundo = {f1_prof:.3f}   |   baseline = {f1_base:.3f}')

    # AUC-ROC (si hay puntajes de probabilidad)
    y_bin = (y == 'fraudulento').astype(int)
    if score_prof is not None:
        print(f'AUC-ROC           profundo = {roc_auc_score(y_bin, score_prof):.3f}', end='')
        if score_base is not None:
            print(f'   |   baseline = {roc_auc_score(y_bin, score_base):.3f}')
        else:
            print()

    # Prueba de McNemar
    tabla, p = prueba_mcnemar(y, pred_prof, pred_base)
    print(f'\nMcNemar: tabla={tabla}  ->  p = {p:.5f}')
    print('  Diferencia SIGNIFICATIVA (p < 0.05): el profundo difiere del baseline.'
          if p < 0.05 else '  Diferencia NO significativa (p >= 0.05).')

    # Criterio F1 >= 0.90
    print(f'\nCriterio F1 >= {F1_MINIMO}: ' +
          ('CUMPLE' if f1_prof >= F1_MINIMO else 'NO cumple') + f' (F1 profundo = {f1_prof:.3f})')

    # Veredicto
    gana = f1_prof > f1_base and p < 0.05
    print('\nVEREDICTO: el modelo profundo SUPERA al baseline de forma significativa.'
          if gana else '\nVEREDICTO: no se puede afirmar superioridad significativa; revisar.')
    matriz(y, pred_prof, f'{nombre} — modelo profundo (test)', f'03_{nombre}_cm_profundo.png')
    return {'f1_prof': f1_prof, 'f1_base': f1_base, 'p': p, 'gana': gana}

## 2. Cargar predicciones sobre TEST (o DEMO)

Con datos reales: carga los modelos guardados y predice sobre el test **una sola vez**.
Ejemplo para el motor de URL (baseline en local):
```python
import joblib
logreg = joblib.load(MODELOS / 'url_logreg.joblib')
test = pd.read_csv(PARTICIONES / 'url_test.csv')
pred_base = logreg.predict(test['url'])
score_base = logreg.predict_proba(test['url'])[:, list(logreg.classes_).index('fraudulento')]
```
(La CNN y BETO se cargan en Colab con torch/transformers y se predice igual sobre su test.)

In [ ]:
def demo(nombre, n, acc_prof, acc_base, nit_prof, nit_base):
    """Genera un test sintético con predicciones de dos modelos, para probar el flujo."""
    y_bin = rng.choice([0, 1], size=n, p=[0.7, 0.3])
    ID = {0: 'legitimo', 1: 'fraudulento'}
    y = np.array([ID[i] for i in y_bin])
    def preds(acc):
        flip = rng.random(n) > acc
        return np.array([ID[i] for i in np.where(flip, 1 - y_bin, y_bin)])
    def scores(nit):
        s = rng.normal(np.where(y_bin == 1, 0.5 + nit, 0.5 - nit), 0.15)
        return np.clip(s, 0, 1)
    return y, preds(acc_prof), preds(acc_base), scores(nit_prof), scores(nit_base)

print('Generando DEMO (reemplaza por tus predicciones reales cuando las tengas).')

## 3. Motor de URL — CNN vs. baseline

In [ ]:
y_url, pred_cnn, pred_url_base, sc_cnn, sc_url_base = demo(
    'url', n=400, acc_prof=0.93, acc_base=0.85, nit_prof=0.28, nit_base=0.15)
res_url = comparar(y_url, pred_cnn, pred_url_base, sc_cnn, sc_url_base, nombre='url')

## 4. Motor de texto — BETO vs. baseline

In [ ]:
y_txt, pred_beto, pred_txt_base, sc_beto, sc_txt_base = demo(
    'texto', n=300, acc_prof=0.92, acc_base=0.83, nit_prof=0.27, nit_base=0.14)
res_txt = comparar(y_txt, pred_beto, pred_txt_base, sc_beto, sc_txt_base, nombre='texto')

## 5. Revisión manual de errores (material para la discusión)

In [ ]:
# Los 20 primeros casos donde el modelo profundo se equivocó (aquí, motor de texto)
errores = pd.DataFrame({'y_real': y_txt, 'pred_beto': pred_beto})
errores = errores[errores['y_real'] != errores['pred_beto']].head(20)
print(f'Errores mostrados: {len(errores)} (con datos reales, incluye la columna del texto)')
errores

## 6. Veredicto de la hipótesis

- Se reportan, para cada motor, exactitud/precisión/recall/F1/AUC + matriz de confusión.
- La **prueba de McNemar** (valor *p*) decide si la mejora del modelo profundo sobre el
  baseline es estadísticamente significativa.
- Se comprueba el criterio **F1 ≥ 0.90**.

> Recuerda: estos números salen del **DEMO**. Con tus datos reales, este mismo cuaderno
> produce el veredicto formal de tu tesis. Guarda las predicciones (no solo las métricas).